In [1]:
from dotenv import  load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState, StateGraph,START,END

load_dotenv(override = True)

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body=
    {
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 状態を宣言
class OverAllState(MessagesState):
    output:str

#2. ノードを宣言
def llm_mode(state:OverAllState) -> OverAllState:
    messages = state["messages"]
    res = model.invoke(messages)
    return {
        "messages":[res]
    }

def output_node(state:OverAllState) -> OverAllState:
    return {
        "output":state["messages"][-1].content
    }

#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node("llm_node",llm_mode)
builder.add_node("output_node",output_node)
builder.add_edge(START,"llm_node")
builder.add_edge("llm_node","output_node")
builder.add_edge("output_node",END)

#4. チェックポイントストレージを設定
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

#5. 使用する際は必ずスレッドIDを指定する
config = {
    "configurable":{
        "thread_id":"chapter03-01"
    }
}

#6. グラフを実行
graph.invoke({"messages":[HumanMessage("こんにちは、私は田中です")]},config=config)


{'messages': [HumanMessage(content='こんにちは、私は田中です', additional_kwargs={}, response_metadata={}, id='533b5eca-0790-49ea-b30d-5783e63d3de2'),
  AIMessage(content='こんにちは、田中さん！お会いできてうれしいです。どのようにお手伝いしましょうか？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 13, 'total_tokens': 41, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'cb03ff15-74e4-4af7-9f2d-1800258edec6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ffff5-5e5b-7253-9a2a-724e52590b04-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 28, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})],
 'output': 'こんにちは、田中さん！お会いできてうれしいです。どの

In [2]:
graph.invoke({"messages":[HumanMessage("こんにちは、私は誰ですか")]},config=config)


{'messages': [HumanMessage(content='こんにちは、私は田中です', additional_kwargs={}, response_metadata={}, id='533b5eca-0790-49ea-b30d-5783e63d3de2'),
  AIMessage(content='こんにちは、田中さん！お会いできてうれしいです。どのようにお手伝いしましょうか？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 13, 'total_tokens': 41, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'cb03ff15-74e4-4af7-9f2d-1800258edec6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ffff5-5e5b-7253-9a2a-724e52590b04-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 28, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  HumanMessage(content='こんにちは、私は誰ですか', 

In [3]:
config1 = {
    "configurable":{
        "thread_id":"chapter03-01xx"
    }
}
graph.invoke({"messages":[HumanMessage("こんにちは、私は誰ですか")]},config=config1)


{'messages': [HumanMessage(content='こんにちは、私は誰ですか', additional_kwargs={}, response_metadata={}, id='ad910c9a-87f2-46c7-84a9-78db690647d3'),
  AIMessage(content='こんにちは！あなたが誰かを教えていただいていないので、私はあなたのことを知りません。もしあなたの名前や身元を教えていただければ、それに基づいてお話しできますよ😊\n\n何かお手伝いできることがあれば、遠慮なく教えてくださいね！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 13, 'total_tokens': 85, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '9ced247a-c18f-434d-964e-f0a3661b1216', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ffff5-6852-7233-88c2-e2dfbb3440eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 72, 'total_tokens': 85, 'input_token_details': {'cache_read': 